# 02. Feature Discovery at the Pre-NPI Position

- **Project:** Mechanistic analysis of negative-polarity-item (NPI) licensing in Gemma-2-2B.
- **Stage:** Representational i.e. which SAE features encode licensing at the position that predicts the NPI?
- **Input:** npi_stimuli.csv
- **Outputs:** 02_validated_features.csv, 02_ablation_targets.csv, 02_features.png
- **Runtime:** ~40 min
- **Summary:** Extracts residual-stream activations at the pre-NPI position (the token whose
logits produce P("any")) and finds 65k-width SAE features that separate licensed from
unlicensed conditions. Discovery uses sentential negation only; all other environments
are held out to test generalization. Feature indices here are 65k-specific.

In [ ]:
!pip install sae_lens transformers accelerate pandas matplotlib seaborn scipy --quiet

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login
login(os.environ["HF_TOKEN"])
print("Authenticated.")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc, torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from transformers import AutoTokenizer, AutoModelForCausalLM
from sae_lens import SAE
import warnings
warnings.filterwarnings('ignore')
gc.collect(); torch.cuda.empty_cache()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
import glob, pandas as pd
csv_path = glob.glob("/kaggle/input/**/npi_stimuli.csv", recursive=True)[0]
stim_df = pd.read_csv(csv_path)
stimuli = list(stim_df[["item","condition","sentence","npi"]].itertuples(index=False, name=None))
item_info = stim_df.drop_duplicates("item").set_index("item")[["environment","design"]].to_dict("index")
print(f"Loaded {len(stimuli)} sentences, {stim_df.item.nunique()} items from {csv_path}")

In [ ]:
MODEL_NAME = "google/gemma-2-2b"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16,
    device_map="auto", low_cpu_mem_usage=True)
model.eval()
print(f"Loaded. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
LAYERS_TO_PROBE = [12, 14, 16, 18, 20, 22]

def find_any_position(sentence, tokenizer):
    inputs = tokenizer(sentence, return_tensors="pt")
    ids = inputs["input_ids"][0]
    for i in range(len(ids)):
        if tokenizer.decode([ids[i].item()]).strip().lower() == "any":
            return i, inputs
    return None, inputs

captured = {}
def make_cap_hook(layer_idx):
    def hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        captured[layer_idx] = h.detach()
    return hook

handles = [model.model.layers[l].register_forward_hook(make_cap_hook(l)) for l in LAYERS_TO_PROBE]

activations = {l: [] for l in LAYERS_TO_PROBE}
metadata = []

for idx, (item, condition, sentence, _) in enumerate(stimuli):
    any_pos, inputs = find_any_position(sentence, tokenizer)
    if any_pos is None:
        print(f"  skip (no 'any'): {sentence}")
        continue
    if any_pos < 1:
        print(f"  skip (any too early): {sentence}")
        continue
    pre_pos = any_pos - 1
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        _ = model(**inputs)
    for l in LAYERS_TO_PROBE:
        activations[l].append(captured[l][0, pre_pos].cpu().float())
    metadata.append({"idx": idx, "item": item, "condition": condition,
                     "sentence": sentence, "any_pos": any_pos, "pre_pos": pre_pos,
                     "environment": item_info[item]["environment"],
                     "design": item_info[item]["design"]})
    if (idx + 1) % 30 == 0:
        print(f"  {idx + 1}/{len(stimuli)}")

for h in handles:
    h.remove()

meta_df = pd.DataFrame(metadata)

# Discovery/validation/generalization masks
meta_df["in_discovery"]      = meta_df["item"].isin(range(1, 16))
meta_df["in_heldout_neg"]    = meta_df["item"].isin(range(16, 26))
meta_df["in_generalization"] = meta_df["environment"] != "sentential_negation"

for l in LAYERS_TO_PROBE:
    activations[l] = torch.stack(activations[l])

print(f"\nCollected {len(meta_df)} sentences × {len(LAYERS_TO_PROBE)} layers at pre-NPI position.")
print(f"  discovery (sent.neg 1-15):     {meta_df['in_discovery'].sum()}")
print(f"  held-out neg (sent.neg 16-25): {meta_df['in_heldout_neg'].sum()}")
print(f"  generalization (other envs):   {meta_df['in_generalization'].sum()}")

del model
gc.collect(); torch.cuda.empty_cache()
print("Model unloaded for SAE encoding.")

In [ ]:
sae_features = {}
for l in LAYERS_TO_PROBE:
    print(f"Layer {l}...")
    sae, _, _ = SAE.from_pretrained(
        release="gemma-scope-2b-pt-res-canonical",
        sae_id=f"layer_{l}/width_65k/canonical", device=device)
    sae.eval()
    with torch.no_grad():
        acts = activations[l].to(device).to(torch.float16)
        sae_features[l] = sae.encode(acts).cpu().float()
    print(f"  {sae_features[l].shape}, avg active: {(sae_features[l]>0).float().sum(1).mean():.0f}")
    del sae, acts
    gc.collect(); torch.cuda.empty_cache()
print("Done.")

In [ ]:
def rank_features(features, a_mask, bc_mask):
    aa = features[a_mask].numpy(); bb = features[bc_mask].numpy()
    ma, mb = aa.mean(0), bb.mean(0)
    va = aa.var(0, ddof=1) + 1e-10; vb = bb.var(0, ddof=1) + 1e-10
    t = (ma - mb) / np.sqrt(va/len(aa) + vb/len(bb))
    return pd.DataFrame({"feature_idx": np.arange(len(ma)), "mean_A": ma,
                          "mean_BC": mb, "t_stat": t, "abs_t": np.abs(t)})

disc_mask = meta_df["in_discovery"].values
disc_meta = meta_df[disc_mask].reset_index(drop=True)
hold_mask = meta_df["in_heldout_neg"].values 
hold_meta = meta_df[hold_mask].reset_index(drop=True)

rankings = {}
for l in LAYERS_TO_PROBE:
    fd = sae_features[l][disc_mask]
    am = (disc_meta["condition"] == "A").values
    bcm = disc_meta["condition"].isin(["B", "C"]).values
    r = rank_features(fd, am, bcm).sort_values("abs_t", ascending=False).reset_index(drop=True)
    rankings[l] = r
    print(f"\nLayer {l} top 5 (discovery)")
    print(r.head(5).to_string(index=False))

K = 20
validated = []
for l in LAYERS_TO_PROBE:
    top = rankings[l].head(K)["feature_idx"].values
    fh = sae_features[l][hold_mask]
    am = (hold_meta["condition"] == "A").values
    bm = (hold_meta["condition"] == "B").values
    cm = (hold_meta["condition"] == "C").values
    for fi in top:
        va = fh[am, fi].numpy(); vb = fh[bm, fi].numpy(); vc = fh[cm, fi].numpy()
        tab, pab = stats.ttest_ind(va, vb, equal_var=False)
        tac, pac = stats.ttest_ind(va, vc, equal_var=False)
        tbc, pbc = stats.ttest_ind(vb, vc, equal_var=False)
        validated.append({"layer": l, "feature_idx": int(fi),
            "mean_A_hold": va.mean(), "mean_B_hold": vb.mean(), "mean_C_hold": vc.mean(),
            "t_AvB_hold": tab, "p_AvB_hold": pab, "t_AvC_hold": tac, "p_AvC_hold": pac,
            "t_BvC_hold": tbc, "p_BvC_hold": pbc})
val_df = pd.DataFrame(validated)


def classify(row, alpha=0.05):
    agb = row["p_AvB_hold"] < alpha and row["mean_A_hold"] > row["mean_B_hold"]
    agc = row["p_AvC_hold"] < alpha and row["mean_A_hold"] > row["mean_C_hold"]
    bnc = row["p_BvC_hold"] < alpha
    alb = row["p_AvB_hold"] < alpha and row["mean_A_hold"] < row["mean_B_hold"]
    alc = row["p_AvC_hold"] < alpha and row["mean_A_hold"] < row["mean_C_hold"]
    if agb and agc and not bnc: return "SCOPE_SPECIFIC"
    if agb and agc and bnc: return "NEGATION_GRADED"
    if alb or alc: return "INVERSE"
    if not agb and not agc: return "WEAK"
    return "PARTIAL"

val_df["pattern"] = val_df.apply(classify, axis=1)
print("PATTERN CLASSIFICATION at pre-NPI position")
print(val_df["pattern"].value_counts())

ss = val_df[val_df["pattern"] == "SCOPE_SPECIFIC"].sort_values("t_AvB_hold", ascending=False)
print("\nSCOPE_SPECIFIC at pre-NPI position")
print(ss.to_string(index=False) if len(ss) else "None at α=0.05")
ng = val_df[val_df["pattern"] == "NEGATION_GRADED"].sort_values("t_AvB_hold", ascending=False)
print("\nNEGATION_GRADED at pre-NPI position")
print(ng.head(10).to_string(index=False) if len(ng) else "None")

In [ ]:
def plot_feature(l, fi, ax):
    vals = sae_features[l][:, fi].numpy()
    dfp = pd.DataFrame({"activation": vals,
                        "condition": meta_df["condition"].values,
                        "in_discovery": meta_df["in_discovery"].values})
    sns.stripplot(data=dfp, x="condition", y="activation", hue="in_discovery",
                  order=["A", "B", "C"], palette={True: "#3498db", False: "#e67e22"},
                  jitter=0.2, alpha=0.7, ax=ax)
    sns.pointplot(data=dfp, x="condition", y="activation", order=["A", "B", "C"],
                  color="black", markers="D", errorbar=("ci", 95), capsize=0.1,
                  linestyles="--", ax=ax)
    ax.set_title(f"L{l} F{fi}", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("SAE activation")
    ax.set_xticklabels(["A\n(licensed)", "B\n(out of scope)", "C\n(unlicensed)"], fontsize=8)
    ax.legend(title="discovery", fontsize=7)

# choose which features to plot
if len(ss) >= 4:
    to_plot = ss.head(6)[["layer", "feature_idx"]].values.tolist()
    title = "Top SCOPE_SPECIFIC features (pre-NPI position)"
elif len(ng) >= 2:
    to_plot = ng.head(6)[["layer", "feature_idx"]].values.tolist()
    title = "Top NEGATION_GRADED features (pre-NPI position)"
else:
    to_plot = [[l, int(rankings[l].iloc[0]["feature_idx"])] for l in LAYERS_TO_PROBE]
    title = "Top feature per layer (fallback)"

n = len(to_plot)
fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
if n == 1:
    axes = [axes]
for ax, (l, fi) in zip(axes, to_plot):
    plot_feature(int(l), int(fi), ax)
fig.suptitle(f"{title}\n(discovery on sentential negation; A/B/C pooled across all environments)",
             fontsize=11)
plt.tight_layout()
plt.savefig("02_features.png", dpi=150, bbox_inches="tight")
plt.show()

# save outputs
val_df.to_csv("02_validated_features.csv", index=False)
cands = (ss.head(5) if len(ss)
         else (ng.head(5) if len(ng)
               else val_df.sort_values("t_AvB_hold", ascending=False).head(5)))
cands.to_csv("02_ablation_targets.csv", index=False)
print("Saved 02_validated_features.csv, 02_ablation_targets.csv, 02_features.png")

# Neuronpedia links
print("\nNeuronpedia links for top candidates:")
for _, r in cands.iterrows():
    l, fi = int(r["layer"]), int(r["feature_idx"])
    print(f"  L{l} F{fi} (A={r['mean_A_hold']:.2f} B={r['mean_B_hold']:.2f} "
          f"C={r['mean_C_hold']:.2f} {r['pattern']})")
    print(f"    https://www.neuronpedia.org/gemma-2-2b/{l}-gemmascope-res-65k/{fi}")

# Generalization: does each candidate separate A vs C in non-negation environments?
print("\n Generalization: candidate A-vs-C separation by environment")
for _, r in cands.iterrows():
    l, fi = int(r["layer"]), int(r["feature_idx"])
    print(f"\nL{l} F{fi} ({r['pattern']}):")
    for env in ["sentential_negation", "no_subject", "few_subject", "question", "without"]:
        em = (meta_df["environment"] == env).values
        a_vals = sae_features[l][em & (meta_df.condition == "A").values, fi]
        c_vals = sae_features[l][em & (meta_df.condition == "C").values, fi]
        a = a_vals.mean().item() if len(a_vals) else float("nan")
        c = c_vals.mean().item() if len(c_vals) else float("nan")
        print(f"  {env:20s} A={a:6.2f}  C={c:6.2f}  gap={a-c:6.2f}")